# Пульсовые ансамбли боковых сборок

Ноутбук строит подписанные ансамблевые формы пульсового сигнала канала 2 для
каждой записи эксперимента 2. Он не оценивает $\Delta\rho_1$ или
$\Delta\rho_2$ и не локализует источник сигнала.

Реальный расчёт принимает только согласованные дыхательные и ЭКГ-sidecar со
статусом `accepted`. Старые файлы `timestamps/*.json` без версии алгоритма и
ручного статуса не используются.


## Построение ансамбля

Для каждого R-зубца сигнал интерполируется на общую временную сетку. Каждый
удар центрируется по среднему уровню до R. Удары за пределами принятого
дыхательного режима, с неполным окном, нечисловыми значениями или принятым
признаком клиппинга исключаются с раздельным указанием причины.

Ансамбли строятся отдельно для задержки после вдоха и задержки после выдоха.
Среднее и стандартная ошибка по ударам характеризуют внутрисессионную
повторяемость. Сердечные циклы одной записи не считаются независимыми
повторениями эксперимента, поэтому эти величины не являются полной
неопределённостью межзаписного сравнения размеров.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np

REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"
ALGORITHM_VERSION = "series33-pulse-ensemble-v1"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def build_ensemble(time_s, signal_ohm, rpeaks_s, interval_s, grid_s, baseline_s, clip=None):
    time_s = np.asarray(time_s, dtype=float)
    signal_ohm = np.asarray(signal_ohm, dtype=float)
    grid_s = np.asarray(grid_s, dtype=float)
    left, right = map(float, interval_s)
    base_left, base_right = map(float, baseline_s)
    if not np.all(np.diff(time_s) > 0) or len(time_s) != len(signal_ohm):
        raise ValueError("Время должно возрастать и совпадать по длине с сигналом")
    baseline_mask = (grid_s >= base_left) & (grid_s <= base_right)
    if baseline_mask.sum() < 2:
        raise ValueError("Базовый интервал должен содержать не менее двух точек")
    beats = []
    rejected = {"outside_mode_or_window": 0, "non_finite": 0, "clipped": 0}
    for rpeak in np.asarray(rpeaks_s, dtype=float):
        sample_times = rpeak + grid_s
        if rpeak < left or rpeak > right or sample_times[0] < time_s[0] or sample_times[-1] > time_s[-1]:
            rejected["outside_mode_or_window"] += 1
            continue
        beat = np.interp(sample_times, time_s, signal_ohm)
        if not np.isfinite(beat).all():
            rejected["non_finite"] += 1
            continue
        if clip is not None:
            low, high, tolerance = clip
            if np.any(beat <= low + tolerance) or np.any(beat >= high - tolerance):
                rejected["clipped"] += 1
                continue
        beat = beat - float(np.mean(beat[baseline_mask]))
        beats.append(beat)
    if len(beats) < 3:
        raise RuntimeError("После QC осталось менее трёх сердечных циклов")
    beats = np.asarray(beats)
    return {
        "mean_ohm": beats.mean(axis=0),
        "se_within_record_ohm": beats.std(axis=0, ddof=1) / np.sqrt(len(beats)),
        "n_beats": int(len(beats)),
        "rejected": rejected,
    }


In [ ]:
time_test = np.arange(0.0, 20.0, 0.002)
rpeaks_test = np.arange(1.0, 19.0, 1.0)
wave_test = 0.020 * np.exp(-((time_test[:, None] - (rpeaks_test[None, :] + 0.25)) / 0.07) ** 2).sum(axis=1)
grid_test = np.arange(-0.15, 0.701, 0.005)
ensemble_test = build_ensemble(time_test, wave_test, rpeaks_test, (0.5, 19.5), grid_test, (-0.12, -0.02))
assert ensemble_test["n_beats"] == len(rpeaks_test)
assert abs(grid_test[int(np.argmax(ensemble_test["mean_ohm"]))] - 0.25) <= 0.005
print("33.03 synthetic_self_test: passed")


In [ ]:
def load_accepted_sidecars(directory, annotation_type):
    records = {}
    for path in sorted(Path(directory).glob("*.json")):
        item = json.loads(path.read_text(encoding="utf-8"))
        if item.get("annotation_type") != annotation_type:
            continue
        if item.get("qc", {}).get("status") != "accepted":
            raise RuntimeError(f"Непринятый {annotation_type} sidecar: {item.get('record_id')}")
        record_id = item.get("record_id")
        if record_id in records:
            raise RuntimeError(f"Повторный record_id: {record_id}")
        records[record_id] = (path, item)
    return records


if not REAL_MODE:
    print("33.03 real_data_status: blocked_until_accepted_breathing_and_ecg_sidecars_and_pulse_calibration")
else:
    config_value = os.environ.get("KALMYKOV_EXP02_CONFIG")
    if not config_value:
        raise RuntimeError("Задайте KALMYKOV_EXP02_CONFIG")
    config_path = Path(config_value).expanduser().resolve()
    config = json.loads(config_path.read_text(encoding="utf-8"))
    data_root = Path(config["data_root"]).expanduser().resolve()
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    pulse = config.get("pulse_analysis", {})
    required = {
        "signal_column", "unit_scale_to_ohm", "sign", "gain",
        "calibration_status", "operator_status", "pre_s", "post_s",
        "grid_step_s", "baseline_s", "hold_margin_s", "clip_levels_input_units",
        "clip_tolerance_input_units", "timing_calibration_status", "group_delay_s",
    }
    if set(pulse) < required:
        raise RuntimeError("В pulse_analysis отсутствуют обязательные поля")
    if pulse["calibration_status"] != "accepted" or pulse["operator_status"] != "delta_impedance_ohm_accepted":
        raise RuntimeError("Не принят оператор перехода RHEO_2 к подписанному ΔZ")
    if pulse["sign"] not in (-1, 1) or float(pulse["gain"]) <= 0 or float(pulse["unit_scale_to_ohm"]) <= 0:
        raise ValueError("Некорректные sign/gain/unit_scale_to_ohm")

    breathing = load_accepted_sidecars(derived_root / "exp02" / "annotations" / "breathing", "breathing")
    ecg = load_accepted_sidecars(derived_root / "exp02" / "annotations" / "ecg", "ecg")
    if set(breathing) != set(ecg):
        raise RuntimeError("Наборы принятых дыхательных и ЭКГ-sidecar различаются")

    import pandas as pd
    grid_s = np.arange(-float(pulse["pre_s"]), float(pulse["post_s"]) + 0.5 * float(pulse["grid_step_s"]), float(pulse["grid_step_s"]))
    clip_raw = pulse["clip_levels_input_units"]
    clip = None
    if clip_raw is not None:
        if not isinstance(clip_raw, list) or len(clip_raw) != 2:
            raise ValueError("clip_levels_input_units должен быть парой или null")
        scale = float(pulse["unit_scale_to_ohm"]) * float(pulse["gain"])
        sign = float(pulse["sign"])
        converted = sorted([sign * scale * float(value) for value in clip_raw])
        clip = (converted[0], converted[1], abs(scale) * float(pulse["clip_tolerance_input_units"]))

    outputs = []
    for record_id in sorted(breathing):
        breathing_path, b = breathing[record_id]
        ecg_path, e = ecg[record_id]
        if b["input"]["sha256"] != e["input"]["sha256"] or e.get("upstream_breathing", {}).get("qc_status") != "accepted":
            raise RuntimeError(f"Нарушена связь дыхания и ЭКГ: {record_id}")
        source_path = (data_root / b["input"]["relative_path"]).resolve()
        source_path.relative_to(data_root)
        if sha256_file(source_path) != b["input"]["sha256"]:
            raise RuntimeError(f"CSV изменился после разметки: {record_id}")
        frame = pd.read_csv(source_path, encoding="utf-8")
        time_s = pd.to_numeric(frame["TIME_s"], errors="raise").to_numpy(dtype=float)
        raw = pd.to_numeric(frame[pulse["signal_column"]], errors="raise").to_numpy(dtype=float)
        signal_ohm = float(pulse["sign"]) * float(pulse["gain"]) * float(pulse["unit_scale_to_ohm"]) * raw
        rpeaks_s = e.get("rpeaks_s")
        if not rpeaks_s:
            raise RuntimeError(f"Нет принятых R-зубцов: {record_id}")
        for mode in ("задержка_вдох", "задержка_выдох"):
            left, right = map(float, b["accepted_modes"][mode])
            margin = float(pulse["hold_margin_s"])
            item = build_ensemble(
                time_s, signal_ohm, rpeaks_s,
                (left + margin, right - margin), grid_s, pulse["baseline_s"], clip,
            )
            outputs.append({
                "record_id": record_id,
                "subject_id": b["subject_id"],
                "size_mm": int(b["size_mm"]),
                "mode": mode,
                "time_from_r_s": grid_s.tolist(),
                "mean_ohm": item["mean_ohm"].tolist(),
                "se_within_record_ohm": item["se_within_record_ohm"].tolist(),
                "n_beats": item["n_beats"],
                "rejected": item["rejected"],
                "input_sha256": b["input"]["sha256"],
                "breathing_sidecar_sha256": sha256_file(breathing_path),
                "ecg_sidecar_sha256": sha256_file(ecg_path),
            })
    expected = int(config["expected_independent_record_count"]) * 2
    if len(outputs) != expected:
        raise RuntimeError(f"Получено {len(outputs)} ансамблей вместо {expected}")
    artifact = {
        "schema_version": 1,
        "analysis": "33.03_pulse_ensembles",
        "algorithm_version": ALGORITHM_VERSION,
        "status": "accepted_input_conditional_ensembles",
        "config_sha256": sha256_file(config_path),
        "signal_operator": {key: pulse[key] for key in (
            "signal_column", "unit_scale_to_ohm", "sign", "gain",
            "calibration_status", "operator_status", "timing_calibration_status", "group_delay_s",
        )},
        "limitations": [
            "se_is_within_record_only",
            "different_sizes_were_recorded_sequentially",
            "ecg_synchrony_does_not_localize_signal_source",
        ],
        "ensembles": outputs,
    }
    out_dir = derived_root / "exp02" / "analysis"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "33.03_ensembles.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("33.03 real_data_status: ensemble_artifact_written", out_path)


## Ограничения интерпретации

Размах `max-min` не используется как основной подписанный сигнал: он стирает
знак и фазу. Синхронность ансамбля с ЭКГ подтверждает временную связь, но не
определяет вклад мягких тканей, лёгкого или более удалённых структур. Такая
декомпозиция является отдельной обратной задачей `33.04`.
